In [1]:
import argparse
import boto3
import sagemaker
from datetime import datetime
from sagemaker.model_monitor import DataCaptureConfig
from sagemaker.model import Model
from sagemaker import Session

# importing monitor requirments:
from sagemaker.model_monitor import (
    DataCaptureConfig,
    DatasetFormat,
    MonitoringOutput,
    ModelQualityMonitor,  # Use this instead of DataQualityMonitor
)

from sagemaker.processing import (
    ProcessingInput,
    ProcessingOutput,
)


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
sess = sagemaker.Session()
bucket = sess.default_bucket()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name
account_id = boto3.client("sts").get_caller_identity().get("Account")
sm_client = boto3.client('sagemaker')
model_package_group_name = f"FERModelGroupName"


In [3]:
# Generate Endpoint Name
endpoint_name = f"FER-Image-Model-{datetime.utcnow():%Y-%m-%d-%H%M}"
print(f"Deploying model to endpoint: {endpoint_name}")

# Initialize SageMaker Session & Role
# sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()

Deploying model to endpoint: FER-Image-Model-2025-02-28-0610


In [4]:
# Define S3 paths
s3_capture_upload_path = f"s3://{bucket}/data-capture/"  # ✅ Define this variable
baseline_dataset_s3_path = f"s3://{bucket}/final_baseline_data/baseline_10_images.csv"
baseline_results_s3_path = f"s3://{bucket}/final_baseline_data/results/"

In [5]:
# s3://sagemaker-us-east-1-399018723364/group-5/models/final_model.tar.gz

In [6]:
# Configure Data Capture
data_capture_config = DataCaptureConfig(
    enable_capture = True,
    sampling_percentage = 100,
    destination_s3_uri = s3_capture_upload_path
)

# Define the Model for Deployment
model_data = f's3://{bucket}/group-5/models/final_model.tar.gz'
image_uri = sagemaker.image_uris.retrieve(
        framework = "pytorch",
        region = region,
        version = "1.10.0",
        image_scope = "inference",
        instance_type = "ml.m5.xlarge"
    )

model = Model(
    image_uri = image_uri,
    model_data = model_data,
    sagemaker_session = sess,
    role = role,
)

# Deploy the Model to an Endpoint 
predictor = model.deploy(
    initial_instance_count=1,
    instance_type = "ml.m4.xlarge",
    endpoint_name = endpoint_name,
    data_capture_config = data_capture_config
)

print(f"Model successfully deployed to endpoint: {endpoint_name}")

-------!Model successfully deployed to endpoint: FER-Image-Model-2025-02-28-0610


In [7]:
# Creating a data quality monitor for Baseline and Schedule
data_quality_monitor = ModelQualityMonitor(
    role = role,
    instance_count = 1,
    instance_type ="ml.m5.xlarge",
    volume_size_in_gb =30,
    max_runtime_in_seconds = 1800,
    base_job_name = "face-express-class-dist-baseline-job", 
    sagemaker_session = sess
)

In [8]:
print("Running baseline job...")

data_quality_monitor.suggest_baseline(
    problem_type="MulticlassClassification",
    baseline_dataset=baseline_dataset_s3_path,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=baseline_results_s3_path,
    ground_truth_attribute="label", 
    inference_attribute="prediction",
    wait=True,
    logs=True
)

print(f"Baseline job completed. Baseline outputs stored at: {baseline_results_s3_path}")

INFO:sagemaker:Creating processing-job with name face-express-class-dist-baseline-job-2025-02-28-06-14-54-506


Running baseline job...
.............2025-02-28 06:16:55.098426: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2025-02-28 06:16:55.098455: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2025-02-28 06:16:56.759954: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory
2025-02-28 06:16:56.759985: W tensorflow/stream_executor/cuda/cuda_driver.cc:269] failed call to cuInit: UNKNOWN ERROR (303)
2025-02-28 06:16:56.760008: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (ip-10-0-224-112.ec2.internal): /proc/driver/nvidia/version does not exist
2025-02-28 

In [9]:
# Define paths for monitoring
baseline_constraints_uri = f"s3://{bucket}/final_baseline_data/results/constraints.json"
monitoring_schedule_name = "cv-class-dist-monitor-schedule"
monitoring_output_uri = f"{baseline_results_s3_path}/monitoring-output" 

In [10]:
from sagemaker.model_monitor import EndpointInput
# Creating a monitoring schedule:

# Define the ground truth S3 location (update this based on your setup)
ground_truth_s3_uri = f"s3://{bucket}/final_ground_truth/ground_truth_10_images.csv"

# Define the problem type 
problem_type = "MulticlassClassification"  # Update this if necessary

print(f"Creating monitoring schedule: {monitoring_schedule_name}")

# ✅ Fix: Specify 'InferenceAttribute' inside `EndpointInput`
endpoint_input = EndpointInput(
    endpoint_name=endpoint_name,
    destination="/opt/ml/processing/input/data",
    probability_attribute=None,   # Set this if you're using probability outputs
    probability_threshold_attribute=None,  # Optional
    inference_attribute="prediction",  # ✅ Column name with predicted class labels in inference data
)

data_quality_monitor.create_monitoring_schedule(
    monitor_schedule_name=monitoring_schedule_name,
    endpoint_input=endpoint_input,
    ground_truth_input=ground_truth_s3_uri, 
    problem_type=problem_type,
    output_s3_uri=monitoring_output_uri,
    constraints=baseline_constraints_uri,  
    schedule_cron_expression="cron(0 * ? * * *)",  
    enable_cloudwatch_metrics=True
)

print(f"Monitoring schedule '{monitoring_schedule_name}' created.")
print(f"Monitoring output will be stored in: {monitoring_output_uri}")

Creating monitoring schedule: cv-class-dist-monitor-schedule


INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: cv-class-dist-monitor-schedule


Monitoring schedule 'cv-class-dist-monitor-schedule' created.
Monitoring output will be stored in: s3://sagemaker-us-east-1-399018723364/final_baseline_data/results//monitoring-output


In [11]:
print("Data drift monitoring for class distribution has been set up successfully.")
print("SageMaker Model Monitor will compare the predicted classes in production")
print("to the baseline distribution and raise alerts if drift occurs.")

Data drift monitoring for class distribution has been set up successfully.
SageMaker Model Monitor will compare the predicted classes in production
to the baseline distribution and raise alerts if drift occurs.
